# ARC 2026 Mounted SGLang Chunk-16 Launcher

Production fork of the 2025 winning-solution launcher. The pinned notebook image supplies the Unsloth training stack, while the retained output of the source notebook supplies the read-only SGLang stack. The launcher uses role-specific Python paths so only inference imports SGLang.

Manual runs use `submit_predictions` on one visible dummy test key. Kaggle competition reruns are automatically promoted to the full `submit_competition` path.


In [ ]:
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"

import unsloth

In [ ]:
MODE = "submit_predictions"  # validation | submit_predictions | submit_competition

CODE_DATASET_ROOT = "/kaggle/input/datasets/yuvraj/arc2026"
PATCH_DATASET_ROOT = "/kaggle/input/datasets/yuvraj/arc2026-mounted-stack-patch"
MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
COMP_ROOT = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
STACK_TARGET = "/kaggle/input/notebooks/yuvraj/arc26-2025-winning-solution-v1-455107/arc_stack"

VALIDATION_KEYS = [
    "0934a4d8",
    "135a2760",
    "136b0064",
    "13e47133",
    "142ca369",
    "16b78196",
    "16de56c4",
    "1818057f",
]  # None = full evaluation set
SMOKE_LIMIT_KEYS = 1

CHUNK_SIZE = 16
TRAIN_NPROCS = 4
INFER_WORKERS = 4
SGLANG_TP_SIZE = 1
USE_SPECULATIVE_DFS = False
SGLANG_SPEC_REPEAT_LEN = 9
SGLANG_DYNAMIC_REPEAT = False
DFS_PROB_THRESHOLD = 0.2
SELECTION_ALGORITHM = "score_kgmon"
PROFILE_TIMINGS = True
KEEP_ADAPTERS = False
VALIDATION_END_TIME_HOURS = 11.5
SUBMIT_PREDICTIONS_END_TIME_HOURS = 0.75
SUBMIT_COMPETITION_END_TIME_HOURS = 11.5
RESET_RUN_ARTIFACTS = True

WORK_NOTEBOOK_ROOT = "/kaggle/working/arc2026_run"
WORK_CODE_DIR = "/kaggle/working/arc2026_run/ARC-AGI1/qwen_baseline"


In [ ]:
import json
import os
from pathlib import Path


def _truthy_env(name: str) -> bool:
    value = os.getenv(name)
    if value is None:
        return False
    return value.lower() not in {"", "0", "false", "no", "off"}


IS_KAGGLE_RERUN = _truthy_env("KAGGLE_IS_COMPETITION_RERUN")
EFFECTIVE_MODE = "submit_competition" if IS_KAGGLE_RERUN else MODE
assert EFFECTIVE_MODE in {"validation", "submit_predictions", "submit_competition"}

EVAL_CHALLENGES = f"{COMP_ROOT}/arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS = f"{COMP_ROOT}/arc-agi_evaluation_solutions.json"
TEST_CHALLENGES = f"{COMP_ROOT}/arc-agi_test_challenges.json"

if EFFECTIVE_MODE == "validation":
    TEST_PATH = EVAL_CHALLENGES
    SOLUTION_PATH = EVAL_SOLUTIONS
    OUTPUT_DIR = "/kaggle/working/inference_outputs_validation"
    STATE_PATH = "/kaggle/working/inference_outputs_validation_chunk_state.json"
    SUBMISSION_PATH = "/kaggle/working/validation_submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = VALIDATION_KEYS
    END_TIME_HOURS = VALIDATION_END_TIME_HOURS
elif EFFECTIVE_MODE == "submit_predictions":
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_submit_smoke"
    STATE_PATH = "/kaggle/working/inference_outputs_submit_smoke_chunk_state.json"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = SMOKE_LIMIT_KEYS
    SELECTED_KEYS = None
    END_TIME_HOURS = SUBMIT_PREDICTIONS_END_TIME_HOURS
else:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_submit"
    STATE_PATH = "/kaggle/working/inference_outputs_submit_chunk_state.json"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = SUBMIT_COMPETITION_END_TIME_HOURS

print("mode_requested =", MODE)
print("kaggle_rerun =", IS_KAGGLE_RERUN)
print("effective_mode =", EFFECTIVE_MODE)
print("test_path =", TEST_PATH)
print("solution_path =", SOLUTION_PATH)
print("output_dir =", OUTPUT_DIR)
print("submission_path =", SUBMISSION_PATH)
print("state_path =", STATE_PATH)
print("selected_keys =", SELECTED_KEYS)
print("limit_keys =", LIMIT_KEYS)
print("end_time_hours =", END_TIME_HOURS)


In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"

assert Path(CODE_DATASET_ROOT).exists(), f"Missing code dataset root: {CODE_DATASET_ROOT}"
assert Path(PATCH_DATASET_ROOT).exists(), f"Missing patch dataset root: {PATCH_DATASET_ROOT}"
assert Path(MODEL_PATH).exists(), f"Missing model path: {MODEL_PATH}"
assert Path(TEST_PATH).exists(), f"Missing challenge path: {TEST_PATH}"
assert Path(STACK_TARGET).is_dir(), f"Missing mounted SGLang stack: {STACK_TARGET}"
assert Path(os.environ["TRITON_PTXAS_PATH"]).exists(), os.environ["TRITON_PTXAS_PATH"]
if SOLUTION_PATH is not None:
    assert Path(SOLUTION_PATH).exists(), f"Missing solution path: {SOLUTION_PATH}"

lora_source = Path(STACK_TARGET) / "sglang" / "srt" / "lora" / "lora.py"
assert lora_source.exists(), f"Missing mounted rsLoRA source: {lora_source}"
assert "self.config.r ** 0.5" in lora_source.read_text(), "Mounted stack lacks the rsLoRA scaling patch"

if RESET_RUN_ARTIFACTS:
    for path in [WORK_NOTEBOOK_ROOT, OUTPUT_DIR, "/kaggle/working/sglang_adapters"]:
        shutil.rmtree(path, ignore_errors=True)
    for path in [STATE_PATH, SUBMISSION_PATH]:
        try:
            Path(path).unlink()
        except FileNotFoundError:
            pass

print("working_disk =")
os.system("df -h /kaggle/working")
print("stack_target =", STACK_TARGET)
print("rslora_patch = verified")
for module_name in ["unsloth", "transformers", "torch", "sglang"]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, spec.origin if spec else "MISSING")


In [ ]:
import os
import shutil
from pathlib import Path

code_root = Path(CODE_DATASET_ROOT)
candidate_paths = [
    code_root / "ARC-AGI1" / "qwen_baseline",
    code_root / "qwen_baseline",
]

src = next((path for path in candidate_paths if path.exists()), None)
if src is None:
    raise FileNotFoundError(f"Could not find qwen_baseline under {code_root}")

dst = Path(WORK_CODE_DIR)
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

patch_root = Path(PATCH_DATASET_ROOT)
expected_commit = (patch_root / "SOURCE_COMMIT.txt").read_text().strip()
assert expected_commit == "771b8cf", f"Unexpected patch commit: {expected_commit}"
for filename in ["arc_sglang.py", "run_chunked_sglang_pipeline.py"]:
    patch_file = patch_root / filename
    assert patch_file.exists(), f"Missing runtime patch: {patch_file}"
    shutil.copy2(patch_file, dst / filename)

assert "ARC_SGLANG_STACK_PATH" in (dst / "arc_sglang.py").read_text()
assert 'phase == "infer"' in (dst / "run_chunked_sglang_pipeline.py").read_text()
os.chdir(dst)

print("cwd =", os.getcwd())
print("copied_from =", src)
print("patched_from =", patch_root)
print("source_commit =", expected_commit)
print("copied_to =", dst)
print("top_files =", sorted(x.name for x in dst.iterdir())[:20])


In [ ]:
import json
import os
import subprocess
import sys
import time

cmd = [
    sys.executable,
    "run_chunked_sglang_pipeline.py",
    "--test-path", TEST_PATH,
    "--model-path", MODEL_PATH,
    "--output-dir", OUTPUT_DIR,
    "--chunk-size", str(CHUNK_SIZE),
    "--train-nprocs", str(TRAIN_NPROCS),
    "--infer-workers", str(INFER_WORKERS),
    "--sglang-tp-size", str(SGLANG_TP_SIZE),
    "--sglang-adapter-dir", "/kaggle/working/sglang_adapters",
    "--sglang-adapter-manifest", "/kaggle/working/sglang_adapters/adapter_manifest.json",
    "--sglang-speculative-repeat-len", str(SGLANG_SPEC_REPEAT_LEN),
    "--dfs-prob-threshold", str(DFS_PROB_THRESHOLD),
    "--selection-algorithm", SELECTION_ALGORITHM,
    "--state-path", STATE_PATH,
    "--submission-path", SUBMISSION_PATH,
    "--end-time", str(time.time() + END_TIME_HOURS * 3600),
]

if USE_SPECULATIVE_DFS:
    cmd.append("--use-speculative-dfs")
if SGLANG_DYNAMIC_REPEAT:
    cmd.append("--sglang-dynamic-repeat")
if PROFILE_TIMINGS:
    cmd.append("--profile-timings")
if KEEP_ADAPTERS:
    cmd.append("--keep-adapters")
if SELECTED_KEYS is not None:
    cmd.extend(["--keys-json", json.dumps(SELECTED_KEYS)])
elif LIMIT_KEYS is not None:
    cmd.extend(["--limit-keys", str(LIMIT_KEYS)])

env = os.environ.copy()
env["ARC_SGLANG_STACK_PATH"] = STACK_TARGET
pythonpath = [p for p in env.get("PYTHONPATH", "").split(os.pathsep) if p and p != STACK_TARGET]
env["PYTHONPATH"] = os.pathsep.join(pythonpath)

print("running:", " ".join(cmd))
print("mounted_sglang_stack =", env["ARC_SGLANG_STACK_PATH"])
subprocess.run(cmd, cwd=WORK_CODE_DIR, env=env, check=True)


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

if WORK_CODE_DIR not in sys.path:
    sys.path.insert(0, WORK_CODE_DIR)

from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon


def _requested_validation_keys(test_path: str, selected_keys):
    if selected_keys is not None:
        return list(selected_keys)
    with open(test_path) as f:
        return sorted(json.load(f).keys())


def _expected_base_subkeys(test_path: str, puzzle_keys):
    with open(test_path) as f:
        raw = json.load(f)
    return {key: {f"{key}_{i}" for i in range(len(raw[key]["test"]))} for key in puzzle_keys}


def _observed_base_subkeys(output_dir: Path):
    if not output_dir.exists():
        return set()
    return {path.name.split(".")[0] for path in output_dir.iterdir() if path.is_file()}


def _completed_validation_keys(test_path: str, output_dir: Path, state_path: str, selected_keys):
    requested_keys = _requested_validation_keys(test_path, selected_keys)
    completed_from_state = set()
    state_file = Path(state_path)
    if state_file.exists():
        with open(state_file) as f:
            state = json.load(f)
        completed_from_state = set(state.get("done_keys", []))

    expected = _expected_base_subkeys(test_path, requested_keys)
    observed = _observed_base_subkeys(output_dir)
    completed_from_outputs = {key for key, base_subkeys in expected.items() if base_subkeys.issubset(observed)}

    completed = []
    for key in requested_keys:
        if key in completed_from_state or key in completed_from_outputs:
            completed.append(key)
    return requested_keys, completed


def _stage_completed_outputs(output_dir: Path, completed_keys: list[str]) -> Path:
    staged_dir = Path("/kaggle/working/validation_scoring_subset")
    shutil.rmtree(staged_dir, ignore_errors=True)
    staged_dir.mkdir(parents=True, exist_ok=True)
    key_set = set(completed_keys)
    for path in output_dir.iterdir():
        if not path.is_file():
            continue
        base_key = path.name.split(".")[0].rsplit("_", 1)[0]
        if base_key in key_set:
            shutil.copy2(path, staged_dir / path.name)
    return staged_dir


if EFFECTIVE_MODE == "validation":
    output_dir = Path(OUTPUT_DIR)
    requested_keys, completed_keys = _completed_validation_keys(TEST_PATH, output_dir, STATE_PATH, SELECTED_KEYS)

    print("requested_validation_keys =", len(requested_keys))
    print("completed_validation_keys =", len(completed_keys))
    print("completion_coverage =", f"{len(completed_keys)}/{len(requested_keys)}")

    if completed_keys:
        scoring_output_dir = _stage_completed_outputs(output_dir, completed_keys)
        data = ArcDataset.from_file(TEST_PATH, keys=completed_keys).load_replies(SOLUTION_PATH)
        decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
        if any(scoring_output_dir.iterdir()):
            decoder.load_decoded_results(str(scoring_output_dir))

        selected = decoder.run_selection_algo(score_kgmon)
        submission = data.get_submission(selected)
        score = data.validate_submission(submission)

        with open(SUBMISSION_PATH, "w") as f:
            json.dump(submission, f)

        print("completed_keys =", completed_keys)
        print("selected keys:", sorted(selected.keys()))
        print("completed_only_score =", score)
        print("submission_path =", SUBMISSION_PATH)
        missing_keys = [key for key in requested_keys if key not in completed_keys]
        print("missing_keys =", missing_keys[:20])
    else:
        with open(SUBMISSION_PATH, "w") as f:
            json.dump({}, f)
        print("No completed validation keys yet.")
        print("submission_path =", SUBMISSION_PATH)
else:
    with open(SUBMISSION_PATH) as f:
        submission = json.load(f)
    print("submission_path =", SUBMISSION_PATH)
    print("num_submission_keys =", len(submission))
    preview_keys = sorted(list(submission.keys()))[: min(3, len(submission))]
    for key in preview_keys:
        print(key, submission[key][:1])

out_dir = Path(OUTPUT_DIR)
if out_dir.exists():
    output_files = sorted(path.name for path in out_dir.iterdir() if path.is_file())
    print("num_output_files =", len(output_files))
    for name in output_files[:20]:
        print(name)
